# Checker v1
Validates rows in `notes.csv` and marks each as **PASS**, **WARNING**, or **ERROR** using the OpenAI API.

## 1. Install dependencies

In [ ]:
!pip install openai pandas

## 2. Imports & API key

In [ ]:
import pandas as pd
from openai import OpenAI

# Paste your OpenAI API key here
client = OpenAI(api_key="YOUR_API_KEY_HERE")

## 3. Load the CSV

In [ ]:
# Update this path to wherever your notes.csv is saved
df = pd.read_csv("data/sources/notes.csv")
print(f"Loaded {len(df)} rows")
df.head()

## 4. Rule-based checks

In [ ]:
REQUIRED_COLUMNS = ["note_id", "claim", "source_file", "page_number", "evidence_snippet", "status"]

def check_row(row):
    issues = []

    # Check for missing required columns in the dataset
    missing_cols = [col for col in REQUIRED_COLUMNS if col not in row.index]
    if missing_cols:
        issues.append(f"ERROR: Missing columns: {missing_cols}")

    # Is the claim present?
    if pd.isna(row.get("claim")) or str(row.get("claim")).strip() == "":
        issues.append("ERROR: claim is missing")

    # Is the source file present?
    if pd.isna(row.get("source_file")) or str(row.get("source_file")).strip() == "":
        issues.append("WARNING: source_file is missing")

    # Is the page number present?
    page = row.get("page_number")
    if pd.isna(page) or str(page).strip() == "":
        issues.append("WARNING: page_number is missing")
    else:
        # Is the page number valid (must be a positive integer)?
        try:
            page_int = int(float(page))
            if page_int <= 0:
                issues.append("ERROR: page_number must be a positive number")
        except (ValueError, TypeError):
            issues.append("ERROR: page_number is not a valid number")

    # Determine overall status
    if any(i.startswith("ERROR") for i in issues):
        status = "ERROR"
    elif any(i.startswith("WARNING") for i in issues):
        status = "WARNING"
    else:
        status = "PASS"

    return status, issues

## 5. Run checks on all rows

In [ ]:
results = []

for _, row in df.iterrows():
    status, issues = check_row(row)
    results.append({
        "note_id": row.get("note_id"),
        "claim": row.get("claim"),
        "checker_status": status,
        "issues": "; ".join(issues) if issues else "None"
    })

results_df = pd.DataFrame(results)
results_df

## 6. Use OpenAI to summarize the findings

In [ ]:
# Build a summary of issues to send to OpenAI
issues_summary = results_df[results_df["checker_status"] != "PASS"][["note_id", "checker_status", "issues"]].to_string(index=False)

prompt = f"""
You are a data quality assistant. Below is a summary of issues found in a research notes CSV file.
Each row has a note_id, a status (WARNING or ERROR), and a description of the issue.

{issues_summary}

Please:
1. Summarize the overall data quality in 2-3 sentences.
2. List the most critical issues to fix first.
3. Suggest one general improvement to prevent these issues in the future.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

## 7. Save results to CSV

In [ ]:
results_df.to_csv("data/outputs/checker_v1_results.csv", index=False)
print("Results saved to data/outputs/checker_v1_results.csv")